# 🌾 YOLOv8m — Weed Detection Training (12 Classes)

**Dataset:** CottonWeedDet12 — 12 weed species  
**Model:** YOLOv8m (medium) — best accuracy/speed tradeoff on Colab T4  
**Classes:** waterhemp, morningglory, ragweed, cocklebur, spurred_anteria, prickly_sida, velvetleaf, palmer_amaranth, redroot_pigweed, johnsongrass, tall_morningglory, sicklepod

---
## ⚡ Quick Start Guide
1. **Runtime → Change runtime type → T4 GPU** ← do this FIRST
2. **Set your dataset folder path in Cell 3** (see the big box below)
3. Run all cells (`Runtime → Run all`)
4. The trained model automatically downloads to your PC at the end

---
## 📂 HOW TO FIND YOUR GOOGLE DRIVE FOLDER PATH

```
Step 1 → Click the 📁 Files icon on the LEFT sidebar of Colab
Step 2 → Open: drive  →  MyDrive
Step 3 → Find your dataset folder (right-click it)
Step 4 → Click "Copy path"
Step 5 → Paste it into Cell 3 where it says PASTE_YOUR_FOLDER_PATH_HERE
```

**Example paths:**
```
/content/drive/MyDrive/wheat_yolo_dataset
/content/drive/MyDrive/datasets/CottonWeedDet12
/content/drive/MyDrive/My Datasets/weed_data
```

> ⚠️ Your dataset folder must have this YOLO structure:
> ```
> your_folder/
>   images/
>     train/   ← .jpg or .png images
>     val/
>   labels/
>     train/   ← .txt annotation files
>     val/
> ```

## Cell 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if 'failed' in result.stdout.lower() or result.returncode != 0:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
print('✅ GPU is ready!')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q ultralytics

import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')

import torch
print(f'✅ PyTorch  : {torch.__version__}')
print(f'✅ CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU      : {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 — Mount Google Drive & Load Dataset Folder

> ### 👇 ONLY THING YOU NEED TO CHANGE IN THIS WHOLE NOTEBOOK:
> Replace `PASTE_YOUR_FOLDER_PATH_HERE` with the actual path of your dataset folder on Google Drive.
>
> **How to find your path:**
> 1. Click the 📁 Files icon on the left sidebar
> 2. Open `drive` → `MyDrive`  
> 3. Right-click your dataset folder → **Copy path**
> 4. Paste it below replacing `PASTE_YOUR_FOLDER_PATH_HERE`

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ╔══════════════════════════════════════════════════════════════╗
# ║  ✏️  PASTE YOUR GOOGLE DRIVE FOLDER PATH BELOW              ║
# ║  Example: '/content/drive/MyDrive/wheat_yolo_dataset'       ║
# ╚══════════════════════════════════════════════════════════════╝
DRIVE_DATASET_DIR = '/content/drive/MyDrive/PASTE_YOUR_FOLDER_PATH_HERE'

# ─── Do NOT change anything below this line ────────────────────
DATASET_DIR = '/content/dataset'   # fast local copy path

if not os.path.isdir(DRIVE_DATASET_DIR):
    raise FileNotFoundError(
        f'❌ Folder not found: {DRIVE_DATASET_DIR}\n'
        f'   Please update DRIVE_DATASET_DIR above with the correct path.'
    )

print(f'✅ Found dataset folder: {DRIVE_DATASET_DIR}')

# Copy dataset to local Colab storage (much faster training than reading from Drive)
if os.path.exists(DATASET_DIR):
    print(f'✅ Local copy already exists at {DATASET_DIR} — skipping copy.')
else:
    print('⏳ Copying dataset from Drive to local Colab storage...')
    print('   (This makes training 3-5x faster than reading directly from Drive)')
    shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'✅ Dataset copied to: {DATASET_DIR}')

# Show structure
print('\n📁 Dataset structure:')
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = '  ' * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')
        if len(files) > 3:
            print(f'{subindent}... ({len(files)} files total)')

## Cell 4 — Reorganize Dataset & Split into Train/Val (80/20)

> Handles the **CottonWeedDet12** folder structure:
> - `weedImages/` → images
> - `annotation_YOLO_txt/` → labels
>
> Reorganizes everything into standard YOLO structure and splits **80% train / 20% val**.
> **Skip if your dataset already has `images/train/` and `labels/train/` folders.**

In [ ]:
import os
import shutil
import random
import glob

# ─── Fallback: define DATASET_DIR if Cell 3 was skipped ─────────────
if 'DATASET_DIR' not in dir():
    DATASET_DIR = '/content/dataset'
    print(f'⚠️  DATASET_DIR not set by Cell 3. Using fallback: {DATASET_DIR}')

# ─── DRIVE_DATASET_DIR fallback (update this to your Drive path) ─────
if 'DRIVE_DATASET_DIR' not in dir():
    DRIVE_DATASET_DIR = '/content/drive/MyDrive/CottonWeedDet12'
    print(f'⚠️  DRIVE_DATASET_DIR not set. Using fallback: {DRIVE_DATASET_DIR}')

# ─── Auto-copy annotation_YOLO_txt from Drive if missing locally ─────
ANN_LOCAL = os.path.join(DATASET_DIR, 'annotation_YOLO_txt')
ANN_DRIVE = os.path.join(DRIVE_DATASET_DIR, 'annotation_YOLO_txt')

if not os.path.exists(ANN_LOCAL):
    if os.path.exists(ANN_DRIVE):
        print(f'⏳ Copying annotation_YOLO_txt from Drive to local Colab...')
        shutil.copytree(ANN_DRIVE, ANN_LOCAL)
        print(f'✅ Copied! {len(os.listdir(ANN_LOCAL))} label files ready.')
    else:
        raise FileNotFoundError(
            f'❌ annotation_YOLO_txt not found!\n'
            f'   Locally checked : {ANN_LOCAL}\n'
            f'   Drive checked   : {ANN_DRIVE}\n'
            f'   Please upload annotation_YOLO_txt to Google Drive first,\n'
            f'   then update DRIVE_DATASET_DIR in Cell 3 to the correct path.'
        )
else:
    print(f'✅ annotation_YOLO_txt already local: {len(os.listdir(ANN_LOCAL))} files')

# ─── Define YOLO output dirs ─────────────────────────────────────────
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, 'images', 'train')
VAL_IMG_DIR   = os.path.join(DATASET_DIR, 'images', 'val')
TRAIN_LBL_DIR = os.path.join(DATASET_DIR, 'labels', 'train')
VAL_LBL_DIR   = os.path.join(DATASET_DIR, 'labels', 'val')

# ─── Skip if already split ───────────────────────────────────────────
already_split = (
    os.path.exists(TRAIN_IMG_DIR) and
    len(glob.glob(f'{TRAIN_IMG_DIR}/*.jpg') + glob.glob(f'{TRAIN_IMG_DIR}/*.png')) > 0
)

if already_split:
    n_train = len(glob.glob(f'{TRAIN_IMG_DIR}/*.jpg') + glob.glob(f'{TRAIN_IMG_DIR}/*.png'))
    n_val   = len(glob.glob(f'{VAL_IMG_DIR}/*.jpg')   + glob.glob(f'{VAL_IMG_DIR}/*.png'))
    print(f'\n✅ Dataset already split: {n_train} train / {n_val} val images. Skipping.')
else:
    print('\n🔍 Dataset contents:', os.listdir(DATASET_DIR))

    # ─── Auto-detect image and label folders ─────────────────────────
    IMG_FOLDER = None
    LBL_FOLDER = None

    for c in ['weedImages', 'images', 'imgs', 'JPEGImages', 'Images']:
        p = os.path.join(DATASET_DIR, c)
        if os.path.isdir(p):
            IMG_FOLDER = p
            break

    for c in ['annotation_YOLO_txt', 'labels', 'annotations', 'Annotations']:
        p = os.path.join(DATASET_DIR, c)
        if os.path.isdir(p):
            LBL_FOLDER = p
            break

    if IMG_FOLDER is None or LBL_FOLDER is None:
        raise FileNotFoundError(
            f'❌ Could not find image/label folders inside {DATASET_DIR}\n'
            f'   Found: {os.listdir(DATASET_DIR)}'
        )

    print(f'✅ Images folder : {IMG_FOLDER}')
    print(f'✅ Labels folder : {LBL_FOLDER}')

    # ─── Gather matched image-label pairs ────────────────────────────
    all_imgs = sorted(
        glob.glob(f'{IMG_FOLDER}/**/*.jpg',  recursive=True) +
        glob.glob(f'{IMG_FOLDER}/**/*.JPG',  recursive=True) +
        glob.glob(f'{IMG_FOLDER}/**/*.jpeg', recursive=True) +
        glob.glob(f'{IMG_FOLDER}/**/*.png',  recursive=True)
    )

    paired, skipped = [], 0
    for img_path in all_imgs:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(LBL_FOLDER, stem + '.txt')
        if os.path.exists(lbl_path):
            paired.append((img_path, lbl_path))
        else:
            skipped += 1

    print(f'\n📊 Total images found  : {len(all_imgs)}')
    print(f'   Matched pairs      : {len(paired)}')
    print(f'   Skipped (no label) : {skipped}')

    if len(paired) == 0:
        # Show sample filenames to help debug
        print('\n🔎 Sample image filenames:')
        for p in all_imgs[:3]: print(f'   {os.path.basename(p)}')
        print('🔎 Sample label filenames:')
        for p in glob.glob(f'{LBL_FOLDER}/*.txt')[:3]: print(f'   {os.path.basename(p)}')
        raise ValueError('❌ No matched image-label pairs found. Filenames must match (e.g. img001.jpg ↔ img001.txt)')

    # ─── 80/20 Split ─────────────────────────────────────────────────
    random.seed(42)
    random.shuffle(paired)
    split_idx   = int(len(paired) * 0.8)
    train_pairs = paired[:split_idx]
    val_pairs   = paired[split_idx:]

    print(f'\n📊 80/20 Split:')
    print(f'   Train : {len(train_pairs)} images')
    print(f'   Val   : {len(val_pairs)}   images')

    # ─── Create YOLO folder structure ────────────────────────────────
    for d in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR]:
        os.makedirs(d, exist_ok=True)

    # ─── Copy files ──────────────────────────────────────────────────
    print('\n⏳ Copying files into YOLO structure (this may take a while)...')
    for img_path, lbl_path in train_pairs:
        shutil.copy(img_path, os.path.join(TRAIN_IMG_DIR, os.path.basename(img_path)))
        shutil.copy(lbl_path, os.path.join(TRAIN_LBL_DIR, os.path.basename(lbl_path)))
    for img_path, lbl_path in val_pairs:
        shutil.copy(img_path, os.path.join(VAL_IMG_DIR, os.path.basename(img_path)))
        shutil.copy(lbl_path, os.path.join(VAL_LBL_DIR, os.path.basename(lbl_path)))

    print('\n✅ Dataset reorganized into YOLO structure:')
    print(f'   {DATASET_DIR}/')
    print(f'   ├── images/')
    print(f'   │   ├── train/  ({len(train_pairs)} images)')
    print(f'   │   └── val/    ({len(val_pairs)} images)')
    print(f'   └── labels/')
    print(f'       ├── train/  ({len(train_pairs)} labels)')
    print(f'       └── val/    ({len(val_pairs)} labels)')

print('\n✅ Cell 4 complete — ready for Cell 5 (data.yaml).')


## Cell 5 — Create data.yaml

In [ ]:
import os
import yaml
import glob

# ─── Fallback: define DATASET_DIR if Cell 3 was skipped ─────────────
if 'DATASET_DIR' not in dir():
    DATASET_DIR = '/content/dataset'

TRAIN_PATH = os.path.join(DATASET_DIR, 'images', 'train')
VAL_PATH   = os.path.join(DATASET_DIR, 'images', 'val')

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f'❌ Train path not found: {TRAIN_PATH}\n'
        f'   Make sure Cell 4 ran successfully to reorganize and split the dataset.'
    )

data_yaml = {
    'path' : DATASET_DIR,
    'train': TRAIN_PATH,
    'val'  : VAL_PATH,
    'nc'   : 12,
    'names': [
        'waterhemp', 'morningglory', 'ragweed', 'cocklebur',
        'spurred_anteria', 'prickly_sida', 'velvetleaf',
        'palmer_amaranth', 'redroot_pigweed', 'johnsongrass',
        'tall_morningglory', 'sicklepod'
    ]
}

DATA_YAML_PATH = '/content/data.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('✅ data.yaml created:')
with open(DATA_YAML_PATH) as f:
    print(f.read())

train_imgs = glob.glob(f'{TRAIN_PATH}/*.jpg') + glob.glob(f'{TRAIN_PATH}/*.png')
val_imgs   = glob.glob(f'{VAL_PATH}/*.jpg')   + glob.glob(f'{VAL_PATH}/*.png')
print(f'📊 Training images  : {len(train_imgs)}')
print(f'📊 Validation images: {len(val_imgs)}')

if len(train_imgs) == 0:
    raise ValueError('❌ No training images found! Run Cell 4 first to split the dataset.')


## Cell 5 — Configure Training Hyperparameters

In [ ]:
# Optimized for Colab T4 (16GB VRAM) + 12-class weed dataset
CONFIG = {
    # ── Model ───────────────────────────────────────────────
    'model'         : 'yolov8m.pt',   # Medium — best accuracy/speed for Colab T4
    'data'          : DATA_YAML_PATH,
    'project'       : '/content/runs/train',
    'name'          : 'yolov8m_weed_v1',

    # ── Core ────────────────────────────────────────────────
    'epochs'        : 100,
    'imgsz'         : 640,
    'batch'         : 16,            # T4 handles batch=16 with m easily
    'device'        : 0,
    'workers'       : 2,

    # ── Learning Rate ───────────────────────────────────────
    'optimizer'     : 'AdamW',
    'lr0'           : 0.01,
    'lrf'           : 0.01,
    'cos_lr'        : True,
    'warmup_epochs' : 3,
    'warmup_momentum': 0.8,
    'momentum'      : 0.937,
    'weight_decay'  : 0.0005,

    # ── Regularization ──────────────────────────────────────
    'dropout'       : 0.0,
    'label_smoothing': 0.1,          # Helps with class imbalance

    # ── Augmentation ────────────────────────────────────────
    'mosaic'        : 1.0,
    'mixup'         : 0.15,
    'copy_paste'    : 0.2,           # Great for rare weed classes
    'fliplr'        : 0.5,
    'flipud'        : 0.1,
    'degrees'       : 10.0,
    'translate'     : 0.1,
    'scale'         : 0.5,
    'hsv_h'         : 0.015,
    'hsv_s'         : 0.7,
    'hsv_v'         : 0.4,
    'erasing'       : 0.4,
    'close_mosaic'  : 10,

    # ── Early Stopping ──────────────────────────────────────
    'patience'      : 20,

    # ── Output ──────────────────────────────────────────────
    'save'          : True,
    'save_period'   : 10,
    'plots'         : True,
    'amp'           : True,
    'cache'         : True,
    'val'           : True,
    'exist_ok'      : True,
}

print('✅ Training config ready:')
for k, v in CONFIG.items():
    print(f'   {k:<22}: {v}')

## Cell 6 — Start Training 🚀

> ⏱️ **Estimated time:** ~90s/epoch × 100 epochs ≈ **2.5 hours** on Colab T4

In [ ]:
from ultralytics import YOLO
import time

print('=' * 60)
print('  🚀 YOLOv8m WEED DETECTION TRAINING')
print('=' * 60)

start_time = time.time()

model = YOLO(CONFIG.pop('model'))
results = model.train(**CONFIG)

elapsed = (time.time() - start_time) / 3600
print(f'\n✅ Training complete in {elapsed:.2f} hours')
print(f'📁 Best weights : {results.save_dir}/weights/best.pt')
print(f'📁 Last weights : {results.save_dir}/weights/last.pt')

## Cell 7 — Evaluate on Validation Set

In [ ]:
best_weights = f'{results.save_dir}/weights/best.pt'
eval_model = YOLO(best_weights)
metrics = eval_model.val(
    data=DATA_YAML_PATH,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    verbose=True,
    plots=True,
)

print('\n' + '=' * 50)
print('  📊 EVALUATION RESULTS')
print('=' * 50)
print(f'  mAP@0.5     : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'  Precision   : {metrics.box.mp:.4f}')
print(f'  Recall      : {metrics.box.mr:.4f}')
print('=' * 50)

class_names = [
    'waterhemp', 'morningglory', 'ragweed', 'cocklebur',
    'spurred_anteria', 'prickly_sida', 'velvetleaf',
    'palmer_amaranth', 'redroot_pigweed', 'johnsongrass',
    'tall_morningglory', 'sicklepod'
]
print('\n  Per-class AP@0.5:')
if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
    for name, ap in zip(class_names, metrics.box.ap50):
        bar = '█' * int(ap * 20)
        print(f'  {name:<22}: {ap:.4f}  {bar}')

## Cell 8 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = os.path.join(str(results.save_dir), 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('YOLOv8m Training Results — Weed Detection', fontsize=16, fontweight='bold')

    plots = [
        ('train/box_loss', 'val/box_loss', 'Box Loss'),
        ('train/cls_loss', 'val/cls_loss', 'Class Loss'),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
        ('metrics/precision(B)', None, 'Precision'),
        ('metrics/recall(B)',    None, 'Recall'),
        ('metrics/mAP50(B)',     None, 'mAP@0.5'),
    ]
    for ax, (train_col, val_col, title) in zip(axes.flatten(), plots):
        if train_col in df.columns:
            ax.plot(df[train_col], label='Train', color='#2196F3')
        if val_col and val_col in df.columns:
            ax.plot(df[val_col],   label='Val',   color='#FF5722')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Training curves saved.')
else:
    print(f'⚠️  results.csv not found at {results_csv}')

## Cell 9 — Save Trained Model Back to Google Drive

In [ ]:
# Saves best.pt back to your Google Drive so you never lose it
DRIVE_SAVE_DIR = '/content/drive/MyDrive/yolo_models/yolov8m_weed_v1'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for fname, src in [
    ('best.pt',              f'{results.save_dir}/weights/best.pt'),
    ('last.pt',              f'{results.save_dir}/weights/last.pt'),
    ('data.yaml',            DATA_YAML_PATH),
    ('training_curves.png',  '/content/training_curves.png'),
    ('results.csv',          os.path.join(str(results.save_dir), 'results.csv')),
]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_SAVE_DIR, fname))
        print(f'✅ Saved {fname} → {DRIVE_SAVE_DIR}')

print(f'\n📁 All files saved to Drive: {DRIVE_SAVE_DIR}')

## Cell 10 — ⬇️ Download Trained Model to Your PC

In [ ]:
from google.colab import files

EXPORT_DIR = '/content/yolov8m_weed_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copy(f'{results.save_dir}/weights/best.pt', f'{EXPORT_DIR}/best.pt')
shutil.copy(f'{results.save_dir}/weights/last.pt', f'{EXPORT_DIR}/last.pt')
shutil.copy(DATA_YAML_PATH,                         f'{EXPORT_DIR}/data.yaml')

if os.path.exists('/content/training_curves.png'):
    shutil.copy('/content/training_curves.png', f'{EXPORT_DIR}/training_curves.png')

results_csv = os.path.join(str(results.save_dir), 'results.csv')
if os.path.exists(results_csv):
    shutil.copy(results_csv, f'{EXPORT_DIR}/results.csv')

with open(f'{EXPORT_DIR}/model_info.txt', 'w') as f:
    f.write('YOLOv8m Weed Detection Model\n')
    f.write('=' * 40 + '\n')
    f.write('Classes (nc=12):\n')
    for i, cls in enumerate(['waterhemp','morningglory','ragweed','cocklebur',
                              'spurred_anteria','prickly_sida','velvetleaf',
                              'palmer_amaranth','redroot_pigweed','johnsongrass',
                              'tall_morningglory','sicklepod']):
        f.write(f'  {i}: {cls}\n')
    f.write('\nUsage:\n')
    f.write('  from ultralytics import YOLO\n')
    f.write('  model = YOLO("best.pt")\n')
    f.write('  results = model.predict("image.jpg", conf=0.25)\n')

zip_path = '/content/yolov8m_weed_model.zip'
shutil.make_archive('/content/yolov8m_weed_model', 'zip', EXPORT_DIR)
print(f'✅ ZIP ready: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('\n⬇️  Starting download to your PC...')
files.download(zip_path)
print('✅ Download triggered!')

## Cell 11 — Quick Inference Test

In [ ]:
import glob
import random
import matplotlib.pyplot as plt

infer_model = YOLO(f'{results.save_dir}/weights/best.pt')

val_images = glob.glob(f'{VAL_PATH}/**/*.jpg', recursive=True) + \
             glob.glob(f'{VAL_PATH}/**/*.png', recursive=True)

if val_images:
    sample = random.sample(val_images, min(4, len(val_images)))
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('YOLOv8m — Sample Predictions on Validation Set', fontsize=14, fontweight='bold')

    for ax, img_path in zip(axes.flatten(), sample):
        pred = infer_model.predict(img_path, conf=0.25, verbose=False)[0]
        result_img = pred.plot()
        ax.imshow(result_img[:, :, ::-1])
        ax.set_title(os.path.basename(img_path), fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Sample predictions saved.')
else:
    print(f'⚠️  No validation images found in {VAL_PATH}')